In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Inductance from the magnetic field energy\n",
    "\n",
    "A circuit designer does not want the field of a coil, but the one number it implies:\n",
    "its inductance. For a single winding carrying the current $I$ it follows from the\n",
    "stored magnetic energy,\n",
    "\n",
    "$$W=\\frac12\\int_\\Omega \\mathbf B\\cdot\\mathbf H\\,\\mathrm dV,\\qquad L=\\frac{2W}{I^2}.$$\n",
    "\n",
    "A homogenised solenoid (inner radius $10$ mm, outer radius $20$ mm, length $120$ mm,\n",
    "$240$ turns, $1\\,\\mathrm A$) is modelled as a uniform azimuthal current density on\n",
    "one explicitly meshed winding region. OneSimulation solves\n",
    "$\\nabla\\times(\\mu^{-1}\\nabla\\times\\mathbf A)=\\mathbf J$ with a weak Coulomb gauge and\n",
    "reconstructs $\\mathbf B=\\nabla\\times\\mathbf A$ and $\\mathbf H=\\mathbf B/\\mu_0$. The\n",
    "energy is then integrated cell by cell over the whole computational domain.\n",
    "\n",
    "**Reference.** The closed-form inductance used here is the long-solenoid formula for a\n",
    "winding of finite radial thickness, multiplied by Nagaoka's end-correction factor:\n",
    "\n",
    "$$L_\\mathrm{ref}=K\\!\\left(\\tfrac{\\ell}{2R_m}\\right)\\,\\mu_0\\,n^2\\,\\ell\\,A_\\mathrm{eff},\\qquad\n",
    "  n=\\frac{N}{\\ell},\\qquad\n",
    "  A_\\mathrm{eff}=\\pi\\Bigl[R_i^2+\\frac{w\\,(R_o+3R_i)}{6}\\Bigr],\\quad w=R_o-R_i .$$\n",
    "\n",
    "$A_\\mathrm{eff}$ is the exact energy-equivalent cross-section of an infinitely long\n",
    "winding in which $B$ falls linearly from $\\mu_0 n I$ at $R_i$ to zero at $R_o$ (for a\n",
    "thin winding it reduces to $\\pi R_m^2$). $K$ is Nagaoka's factor for a current sheet of\n",
    "mean radius $R_m$, written with complete elliptic integrals; for the present\n",
    "length-to-diameter ratio of 4 it is $K\\approx0.90$. Combining the two corrections\n",
    "multiplicatively is accurate to about 2 % for this winding.\n",
    "\n",
    "Two effects make the finite-element inductance fall below $L_\\mathrm{ref}$, and both\n",
    "are quantified below: the magnetic insulation on the finite domain confines the return\n",
    "flux (about 3 %, estimated by a coaxial-return formula), and the finite-element energy\n",
    "of a vector-potential formulation is a lower bound that approaches the exact value\n",
    "from below as the winding is resolved more finely (a further 5-7 % on the modest mesh\n",
    "used here). The example tolerance of 12 % against the free-space closed form, and 10 %\n",
    "once the confinement estimate is taken out, covers both."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "from importlib.metadata import version\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from scipy.special import ellipe, ellipk\n",
    "\n",
    "from onesim.cad.d3 import Cylinder\n",
    "from onesim.materials import VacuumPermeability\n",
    "from onesim.meshing import TetrahedralMesher, LocalRefinement\n",
    "from onesim.numerics import Numerics\n",
    "from onesim.numerics.fem import FiniteElement, Nedelec\n",
    "from onesim.numerics.linear import SparseDirect\n",
    "from onesim.physics.magnetostatics import (\n",
    "    CoulombGauge, MagneticInsulation, MagneticVectorPotential,\n",
    "    Magnetostatics, SolenoidCurrent,\n",
    ")\n",
    "from onesim.simulation import Problem, Simulation\n",
    "\n",
    "print(\"OneSimulation:\", version(\"onesimulation\"))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Parameters and closed-form reference"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "domain_length, domain_radius = 0.32, 0.08       # m, finite computational cylinder\n",
    "coil_length = 0.12                              # m, solenoid length\n",
    "inner_radius, outer_radius = 0.010, 0.020       # m, homogenised winding annulus\n",
    "turns, current = 240, 1.0                       # -, A\n",
    "center_z = domain_length / 2.0                  # coil centred in the domain\n",
    "\n",
    "mu0 = float(VacuumPermeability().evaluate())\n",
    "turns_per_metre = turns / coil_length                                     # n\n",
    "winding_thickness = outer_radius - inner_radius                           # w\n",
    "mean_radius = 0.5 * (inner_radius + outer_radius)                         # R_m\n",
    "B_ideal = mu0 * turns_per_metre * current                                 # mu0 n I\n",
    "area_eff = np.pi * (inner_radius**2 + winding_thickness * (outer_radius + 3 * inner_radius) / 6.0)\n",
    "\n",
    "# Nagaoka's end-correction factor for a current sheet (complete elliptic integrals).\n",
    "k2 = (2 * mean_radius) ** 2 / ((2 * mean_radius) ** 2 + coil_length**2)\n",
    "nagaoka = 4.0 / (3.0 * np.pi * np.sqrt(1.0 - k2)) * (\n",
    "    ((1.0 - k2) / k2) * (ellipk(k2) - ellipe(k2)) + ellipe(k2) - np.sqrt(k2))\n",
    "L_long = mu0 * turns_per_metre**2 * coil_length * area_eff   # infinitely long, thick winding\n",
    "L_ref = nagaoka * L_long                                       # finite length (Nagaoka)\n",
    "\n",
    "# Coaxial-return estimate of the flux confinement by the magnetic insulation at r = R_d.\n",
    "area_return = np.pi * domain_radius**2 - area_eff\n",
    "confinement = 1.0 / (1.0 + area_eff / area_return)\n",
    "\n",
    "print(f\"effective winding area A_eff = {area_eff*1e6:.1f} mm^2  (thin-sheet pi*R_m^2 = {np.pi*mean_radius**2*1e6:.1f} mm^2)\")\n",
    "print(f\"Nagaoka factor K(l/2R_m = {coil_length/(2*mean_radius):.1f}) = {nagaoka:.4f}\")\n",
    "print(f\"closed-form inductance: L_ref = K * mu0 n^2 l A_eff = {L_ref*1e6:.1f} uH\")\n",
    "print(f\"flux-confinement estimate for the finite domain: factor {confinement:.3f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Model, mesh and solver\n",
    "\n",
    "Every face of the cylindrical domain carries magnetic insulation. The winding annulus\n",
    "is an embedded mesh region with a local target size small enough for two to three\n",
    "tetrahedra across the winding thickness, where $B$ falls off linearly."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "geometry = Cylinder(radius=domain_radius, length=domain_length)\n",
    "coil = SolenoidCurrent(\n",
    "    inner_radius=inner_radius, outer_radius=outer_radius, length=coil_length,\n",
    "    center=(0.0, 0.0, center_z), turns=turns, current=current,\n",
    ")\n",
    "model = Magnetostatics(\n",
    "    domain=geometry.volume, permeability=VacuumPermeability(),\n",
    "    formulation=MagneticVectorPotential(gauge=CoulombGauge()), current_sources=(coil,),\n",
    ")\n",
    "problem = Problem(\n",
    "    name=\"Solenoid inductance from field energy\",\n",
    "    geometry=geometry, physics=(model,),\n",
    "    boundary_conditions=(MagneticInsulation(boundary=geometry.boundaries.all, physics=model),),\n",
    ")\n",
    "mesh = TetrahedralMesher(\n",
    "    target_size=0.03, embedded_regions=(coil.support,),\n",
    "    refinements=(LocalRefinement(region=coil.support, target_size=0.0033),),\n",
    ").generate(geometry)\n",
    "print(\"Mesh:\", f\"{mesh.cell_count} tetrahedra\")\n",
    "result = Simulation(\n",
    "    problem=problem, mesh=mesh,\n",
    "    numerics=Numerics(spatial=FiniteElement(element=Nedelec(order=1)), linear_solver=SparseDirect()),\n",
    ").solve_stationary()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Field energy and inductance\n",
    "\n",
    "With first-order Nedelec elements $\\mathbf B=\\nabla\\times\\mathbf A$ is constant on\n",
    "every tetrahedron, so a one-point (centroid) rule integrates $\\mathbf B\\cdot\\mathbf H$\n",
    "exactly. The same rule applied to $\\tfrac12\\int\\mathbf J\\cdot\\mathbf A\\,\\mathrm dV$\n",
    "over the winding cells (flux-linkage form of the energy) must give the same value: for\n",
    "the Galerkin solution the two energy forms coincide."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "B_field = result.field(model.magnetic_flux_density)\n",
    "H_field = result.field(model.magnetic_field_strength)\n",
    "A_field = result.field(model.vector_potential)\n",
    "\n",
    "vertices = np.asarray(mesh.coordinates)[np.asarray(mesh.cells)]           # (cells, 4, 3)\n",
    "edges = vertices[:, 1:] - vertices[:, :1]\n",
    "cell_volumes = np.abs(np.einsum(\"ij,ij->i\", edges[:, 0], np.cross(edges[:, 1], edges[:, 2]))) / 6.0\n",
    "centroids = vertices.mean(axis=1)\n",
    "print(f\"Quadrature: {len(centroids)} cell centroids, meshed volume {cell_volumes.sum()*1e3:.3f} dm^3 \"\n",
    "      f\"(cylinder {np.pi*domain_radius**2*domain_length*1e3:.3f} dm^3)\")\n",
    "\n",
    "B_cells = np.asarray(B_field.evaluate(centroids))\n",
    "H_cells = np.asarray(H_field.evaluate(centroids))\n",
    "energy_cells = 0.5 * np.einsum(\"ij,ij->i\", B_cells, H_cells) * cell_volumes\n",
    "W_energy = float(energy_cells.sum())\n",
    "L_energy = 2.0 * W_energy / current**2\n",
    "\n",
    "# Flux-linkage form: J is the uniform azimuthal winding current density.\n",
    "radius = np.hypot(centroids[:, 0], centroids[:, 1])\n",
    "inside_length = np.abs(centroids[:, 2] - center_z) < coil_length / 2.0\n",
    "in_core = (radius < inner_radius) & inside_length\n",
    "in_winding = (radius >= inner_radius) & (radius < outer_radius) & inside_length\n",
    "J_magnitude = turns * current / (winding_thickness * coil_length)\n",
    "J_cells = np.zeros_like(centroids)\n",
    "J_cells[in_winding, 0] = -J_magnitude * centroids[in_winding, 1] / radius[in_winding]\n",
    "J_cells[in_winding, 1] = J_magnitude * centroids[in_winding, 0] / radius[in_winding]\n",
    "A_cells = np.asarray(A_field.evaluate(centroids[in_winding]))\n",
    "W_linkage = 0.5 * float(np.sum(np.einsum(\"ij,ij->i\", J_cells[in_winding], A_cells) * cell_volumes[in_winding]))\n",
    "L_linkage = 2.0 * W_linkage / current**2\n",
    "\n",
    "# Centre field: same closed form with the computed centre field in place of mu0 n I.\n",
    "B_centre = np.asarray(B_field.evaluate(np.array([[0.0, 0.0, center_z]])))[0]\n",
    "centre_axial = float(B_centre[2])\n",
    "L_centre = nagaoka * turns * centre_axial * area_eff / current\n",
    "\n",
    "share = {name: float(energy_cells[mask].sum()) / W_energy\n",
    "         for name, mask in ((\"core\", in_core), (\"winding\", in_winding), (\"outside\", ~(in_core | in_winding)))}\n",
    "\n",
    "print(f\"stored energy: {W_energy*1e6:.2f} uJ  (core {100*share['core']:.0f} %, winding {100*share['winding']:.0f} %, \"\n",
    "      f\"outside {100*share['outside']:.0f} %)\")\n",
    "print(f\"L from field energy  2W/I^2:          {L_energy*1e6:.1f} uH\")\n",
    "print(f\"L from flux linkage  int J.A dV / I^2: {L_linkage*1e6:.1f} uH\")\n",
    "print(f\"L closed form  K mu0 n^2 l A_eff:      {L_ref*1e6:.1f} uH\")\n",
    "print(f\"centre field: {centre_axial*1e3:.3f} mT = {centre_axial/B_ideal:.3f} mu0 n I;  \"\n",
    "      f\"closed form with this centre field: {L_centre*1e6:.1f} uH\")\n",
    "\n",
    "ratio_ref = L_energy / L_ref\n",
    "ratio_confined = L_energy / (L_ref * confinement)\n",
    "ratio_centre = L_energy / L_centre\n",
    "ratio_routes = abs(L_energy - L_linkage) / L_energy\n",
    "print(f\"L_energy / L_ref                      = {ratio_ref:.3f}   (free-space closed form; FEM lies below it)\")\n",
    "print(f\"L_energy / (L_ref * confinement)      = {ratio_confined:.3f}   (finite-domain confinement taken out)\")\n",
    "print(f\"L_energy / L_centre-based closed form = {ratio_centre:.3f}   (confinement bias removed, mesh error remains)\")\n",
    "print(f\"energy route vs flux-linkage route: {100*ratio_routes:.2f} %\")\n",
    "\n",
    "assert np.isfinite(W_energy) and W_energy > 0, \"The stored energy must be finite and positive.\"\n",
    "assert centre_axial > 0, \"The axial field inside the solenoid must point along +z.\"\n",
    "assert ratio_routes < 0.02, \"Energy and flux-linkage forms of the stored energy disagree by more than 2 %.\"\n",
    "assert 0.88 < ratio_ref < 1.02, \"Inductance deviates from the closed-form reference by more than the example tolerance (12 %).\"\n",
    "assert 0.90 < ratio_confined < 1.02, \"Inductance deviates from the confinement-corrected closed form by more than 10 %.\"\n",
    "assert 0.92 < ratio_centre < 1.02, \"Inductance deviates from the centre-field closed form by more than 8 %.\"\n",
    "assert 0.3 < share[\"winding\"] < 0.6, \"For this thick winding a large share of the energy must sit in the winding.\""
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Figure"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "execution_count": null,
   "outputs": [],
   "source": [
    "cream = \"#fbf5eb\"\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)\n",
    "fig.patch.set_facecolor(cream)\n",
    "for ax in (ax1, ax2):\n",
    "    ax.set_facecolor(\"#fffaf2\"); ax.grid(alpha=0.25)\n",
    "\n",
    "r_line = np.linspace(0.0, 1.5 * outer_radius, 61)\n",
    "Bz_line = np.asarray(B_field.evaluate(np.column_stack((r_line, np.zeros_like(r_line), np.full_like(r_line, center_z)))))[:, 2]\n",
    "Bz_ideal = np.where(r_line < inner_radius, 1.0, np.clip((outer_radius - r_line) / winding_thickness, 0.0, 1.0)) * B_ideal\n",
    "ax1.plot(r_line * 1e3, Bz_ideal * 1e3, color=\"#214D43\", lw=1.8, ls=\"--\", label=\"infinite solenoid (linear fall-off in the winding)\")\n",
    "ax1.plot(r_line * 1e3, Bz_line * 1e3, \"o-\", color=\"#A85539\", ms=3.5, lw=1, label=\"OneSimulation 3D FEM, mid-plane\")\n",
    "ax1.axvspan(inner_radius * 1e3, outer_radius * 1e3, color=\"#c9b89a\", alpha=0.3, label=\"winding\")\n",
    "ax1.set(xlabel=\"Radius r / mm\", ylabel=\"Axial flux density $B_z$ / mT\", title=\"Mid-plane radial profile\")\n",
    "ax1.legend(fontsize=8)\n",
    "\n",
    "labels = (\"field energy\\n$2W/I^2$\", \"flux linkage\\n$\\\\int J\\\\cdot A\\\\,dV/I^2$\",\n",
    "          \"closed form\\n$K\\\\,\\\\mu_0 n^2 \\\\ell A_\\\\mathrm{eff}$\", \"closed form\\nwith FEM centre field\")\n",
    "values = np.array([L_energy, L_linkage, L_ref, L_centre]) * 1e6\n",
    "ax2.bar(labels, values, color=(\"#A85539\", \"#3887c5\", \"#214D43\", \"#6b8e23\"), width=0.6)\n",
    "for i, v in enumerate(values):\n",
    "    ax2.text(i, v * 1.01, f\"{v:.0f} uH\", ha=\"center\", va=\"bottom\", fontsize=9)\n",
    "ax2.set_ylim(0, 1.15 * values.max())\n",
    "ax2.set(ylabel=\"Inductance / uH\", title=\"Inductance: field routes versus closed form\")\n",
    "ax2.tick_params(axis=\"x\", labelsize=8)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Going further\n",
    "\n",
    "Refine the winding (smaller local target size) and watch the energy-based inductance\n",
    "climb towards the reference from below; enlarge the domain radius and the confinement\n",
    "factor approaches one. The same recipe, with $\\tfrac12\\int\\varepsilon|\\mathbf E|^2\\,\\mathrm dV$,\n",
    "extracts capacitances from electrostatic solutions."
   ]
  }
 ],
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "OneSimulation (venv)",
   "language": "python",
   "name": "onesim"
  },
  "language_info": {
   "name": "python"
  }
 }
}